# 04 — Long jobs: checkpointing and scaling

Workloads with a long-job requirement: their needed run time exceeds what any queue allows for that job shape,
they were described as *one calculation longer than a job may run*, or they answered the *Long runs* panel (Section D).
The remedies are considered in order: **checkpoint/restart → run wider → longer wall time**.

*Principles: P3, P4 in the [analysis framework](https://github.com/svaradh/hpc-questionnaire/blob/main/docs/analysis-framework.md). Nothing here scores or ranks groups; flags are candidates for the committee to look at. Totals cover responding groups only, and "Don't know" is never counted as zero; a yes/no flag shown as `<NA>` means its inputs are unknown. Only the respondents' answers and the cluster configuration are used — nothing depends on which code a workload runs.*

**To run:** paste the Spreadsheet ID in the first code cell, then *Runtime → Run all*. Colab asks once per session to sign in with your IISER Google account.

In [ ]:
SPREADSHEET_ID = 'PASTE_YOUR_SPREADSHEET_ID_HERE'   # <- the long string in the sheet URL
REPO_REF = 'main'                                   # git branch/tag for the helper code and cluster config

import os, urllib.request
if not os.path.exists('hpc_analysis.py'):            # in Colab: fetch the helper modules once per session
    for f in ['sheets_client.py', 'hpc_analysis.py', 'hpc_plots.py', 'code_capabilities.json']:
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/svaradh/hpc-questionnaire/{REPO_REF}/analysis/{f}', f)

import pandas as pd
from sheets_client import load_data, load_config
import hpc_analysis as ha
import hpc_plots as hp
pd.set_option('display.max_colwidth', 120)

dfs = load_data(SPREADSHEET_ID)
cfg = load_config(ref=REPO_REF)
wl = ha.workloads(dfs, cfg)
cov = ha.coverage(dfs)
print(f"{cov['groups']} groups responded; {cov['workloads']} workloads described "
      f"({cov['groups_with_workloads']} groups with at least one).")

In [ ]:
lj = ha.long_jobs(wl)
print(f'{len(lj)} workloads with a long-job requirement')
lj

## Split 1 — checkpointing

- *checkpoint available*: the requirement can usually be met by chaining jobs.
- *no checkpoint*: a candidate genuine requirement (see split 2).
- *unknown*: not tested or not known — possibly **an awareness gap**, best met with support rather than policy. Whether the code itself can restart is for the committee to check by hand.

In [ ]:
hp.counts(lj['checkpoint_class'], 'Long-job workloads by checkpoint status')

## Split 2 — scaling (excluding workloads that can checkpoint)

- *high-scaling* (wall time ≤ 0.8× when the cores are doubled): the work could run **wider instead of longer** — the limit that binds is node count, not wall time (see notebook 05).
- *low-scaling*: more nodes do not help — **time, not nodes**, is the need.
- *scaling unknown*: never tested; a scaling test is the next step.

In [ ]:
hp.counts(lj['scaling_split'], 'Long-job workloads by scaling (checkpoint cases excluded)')

## Suggested long-wall-time test

Needed run time > applicable limit **and** no checkpoint **and** not high-scaling **and** evidence level ≤ 3
(see section 4 of the framework). Workloads meeting it are candidates for a long-wall-time service; the committee decides.
`<NA>` = a part of the test is unknown (e.g. checkpoint not tested, evidence source not given): a follow-up, not a failed test.

In [ ]:
display(lj[lj['long_walltime_test'].fillna(False)])
print('Test undecided (inputs unknown):')
lj[lj['long_walltime_test'].isna()]